# FAseg — Ablation on Ex-Vivo Data (Pretraining Only, No Fine-tuning)

Computes per-source FA-boundary metrics for **5 pretraining experiments
× best_s3 checkpoint** against ex-vivo manual annotations, evaluated on the
**UNSEEN** split only.

**No fine-tuning applied** — pretraining `best_s3` models evaluated directly.

**Data split** (same as fine-tuning, for fair comparison):
- **UNSEEN**: src 1–127 + 385–511 (held out)  ← evaluated here

## Table produced (1 total)
| Table | Split |
|-------|-------|
| 1 | UNSEEN |

Results loaded from `outputs/inference_results/exvivo/without_fine_tuning/`.


In [1]:
import os, sys
import numpy as np
from scipy.ndimage import zoom

_NB_DIR = os.path.dirname(os.path.abspath(''))
if os.path.basename(_NB_DIR) == 'scripts':
    _project_root = os.path.dirname(_NB_DIR)
else:
    _project_root = _NB_DIR

MANUAL_DIR  = os.path.join(_project_root, 'manual segmentation', 'manual_seg_beef')
INFER_DIR   = os.path.join(_project_root, 'outputs', 'inference_results', 'exvivo', 'without_fine_tuning')

COLUMN_DT = 2.4e-7

EXPERIMENTS = [
    ('pretraining',                'Full'),
    ('pretraining_no_warmup',      'No Warmup'),
    ('pretraining_no_tof',         'No TOF'),
    ('pretraining_no_domainrand',  'No DomainRand'),
    ('pretraining_no_aug',         'No Aug'),
]

STRATEGY = 'best_s3'

# Same split as fine-tuning for fair comparison
TRAIN_MIN_SRC = 128
TRAIN_MAX_SRC = 384

print(f'Manual masks : {MANUAL_DIR}')
print(f'Inference    : {INFER_DIR}')
print(f'Strategy     : {STRATEGY}  (pretraining only, no fine-tuning)')
print(f'Column dt    : {COLUMN_DT:.1e} s')
print(f'Data split   : SEEN=[{TRAIN_MIN_SRC},{TRAIN_MAX_SRC}]  UNSEEN=rest')

Manual masks : /data/projects/AgentWork/FAseg for github/manual segmentation/manual_seg_beef
Inference    : /data/projects/AgentWork/FAseg for github/outputs/inference_results/exvivo/without_fine_tuning
Strategy     : best_s3  (pretraining only, no fine-tuning)
Column dt    : 2.4e-07 s
Data split   : SEEN=[128,384]  UNSEEN=rest


In [2]:
# =========================================================================
# Load all ex-vivo manual masks
# =========================================================================
def load_exvivo_mask(mask_path):
    """Load mask and process to (384,384) following RealLimbDataset2 logic."""
    mask = np.load(mask_path)
    mask = np.transpose(mask)
    target_height = 4955 // 6  # 826
    scale_factor = target_height / mask.shape[1]
    mask_resized = zoom(mask, (1, scale_factor), order=0)
    mask = mask_resized[0:384, 300:684]
    return mask.astype(bool)

exvivo_masks = {}
for f in sorted(os.listdir(MANUAL_DIR)):
    if f.endswith('_mask.npy'):
        src = int(f.replace('src', '').replace('_mask.npy', ''))
        exvivo_masks[src] = load_exvivo_mask(os.path.join(MANUAL_DIR, f))

print(f'Ex-vivo masks: {len(exvivo_masks)} loaded  (src {min(exvivo_masks.keys())}–{max(exvivo_masks.keys())})')
print(f'  Shape: {exvivo_masks[list(exvivo_masks.keys())[0]].shape}')
print(f'  SEEN  (src {TRAIN_MIN_SRC}–{TRAIN_MAX_SRC}): {sum(1 for s in exvivo_masks if TRAIN_MIN_SRC <= s <= TRAIN_MAX_SRC)} sources')
print(f'  UNSEEN (src 1–{TRAIN_MIN_SRC-1} + {TRAIN_MAX_SRC+1}–511): {sum(1 for s in exvivo_masks if s < TRAIN_MIN_SRC or s > TRAIN_MAX_SRC)} sources')

Ex-vivo masks: 255 loaded  (src 1–511)
  Shape: (384, 384)
  SEEN  (src 128–384): 127 sources
  UNSEEN (src 1–127 + 385–511): 128 sources


In [3]:
# =========================================================================
# Load all inference results
# =========================================================================
infer_results = {}
for exp_key, exp_label in EXPERIMENTS:
    key = f'{exp_key}_{STRATEGY}'
    path = os.path.join(INFER_DIR, key, 'pred_mask_3d.npy')
    if os.path.exists(path):
        infer_results[key] = np.load(path).astype(bool)
        print(f'{key:35s}  loaded')
    else:
        print(f'{key:35s}  MISSING — {path}')

pretraining_best_s3                  loaded
pretraining_no_warmup_best_s3        loaded
pretraining_no_tof_best_s3           loaded
pretraining_no_domainrand_best_s3    loaded
pretraining_no_aug_best_s3           loaded


In [4]:
# =========================================================================
# Metric functions
# =========================================================================
def tof_boundary(mask_2d):
    """Return column index of first '1' per row.  NaN if no TOF."""
    H = mask_2d.shape[0]
    cols = np.full(H, np.nan)
    for r in range(H):
        ones = np.where(mask_2d[r, :])[0]
        if len(ones) > 0:
            cols[r] = float(ones[0])
    return cols


def compute_metrics(manual_2d, pred_2d, column_dt):
    """Return dict of metrics for one (manual, pred) pair."""
    m_bdry = tof_boundary(manual_2d)
    p_bdry = tof_boundary(pred_2d)

    valid = ~np.isnan(m_bdry) & ~np.isnan(p_bdry)
    if valid.sum() == 0:
        return {'max_err_s': np.nan, 'mae_s': np.nan, 'mean_err_s': np.nan,
                'var_s2': np.nan,
                'precision': np.nan, 'recall': np.nan, 'f1': np.nan,
                'n_rows': 0}

    errors = (p_bdry[valid] - m_bdry[valid]) * column_dt
    max_err = np.abs(errors).max()
    mae     = np.abs(errors).mean()
    bias    = errors.mean()
    var_err = errors.var()

    inter = (manual_2d & pred_2d).sum()
    precision = inter / pred_2d.sum()  if pred_2d.sum() > 0  else 0.0
    recall    = inter / manual_2d.sum() if manual_2d.sum() > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        'max_err_s': max_err, 'mae_s': mae, 'mean_err_s': bias, 'var_s2': var_err,
        'precision': precision, 'recall': recall, 'f1': f1,
        'n_rows': int(valid.sum()),
    }

In [5]:
# =========================================================================
# Compute all metrics
# =========================================================================
ALL_METRICS = {}  # key: infer_key → list of per-source metric dicts

for infer_key, infer_3d in infer_results.items():
    per_src = []
    for src in sorted(exvivo_masks.keys()):
        manual_2d = exvivo_masks[src]
        pred_2d   = infer_3d[:, :, src - 1]
        m = compute_metrics(manual_2d, pred_2d, COLUMN_DT)
        m['src'] = src
        per_src.append(m)
    ALL_METRICS[infer_key] = per_src

print('All metrics computed.')

All metrics computed.


In [6]:
# =========================================================================
# Aggregate & print — with ALL / SEEN / UNSEEN splits
# =========================================================================
# Fine-tuning was trained on src 128–384 (mid-half).
# SEEN  = src in [128, 384]
# UNSEEN = src < 128 or src > 384
TRAIN_MIN_SRC = 128
TRAIN_MAX_SRC = 384


def filter_src(per_src, group):
    """Return subset of per_src matching *group* ('all', 'seen', 'unseen')."""
    if group == 'all':
        return per_src
    elif group == 'seen':
        return [s for s in per_src if TRAIN_MIN_SRC <= s['src'] <= TRAIN_MAX_SRC]
    elif group == 'unseen':
        return [s for s in per_src if s['src'] < TRAIN_MIN_SRC or s['src'] > TRAIN_MAX_SRC]
    else:
        raise ValueError(f'Unknown group: {group}')


def aggregate(per_src):
    """Average across sources (macro-average), ignoring NaN."""
    keys = ['max_err_s', 'mae_s', 'mean_err_s', 'var_s2', 'precision', 'recall', 'f1']
    agg = {}
    for k in keys:
        vals = np.array([s[k] for s in per_src if not np.isnan(s[k])])
        agg[k] = vals.mean() if len(vals) > 0 else np.nan
    agg['n_src'] = len(per_src)
    return agg


def print_table(title, strategy, experiments, all_metrics, group='all'):
    """Print one table for a given strategy + data split group."""
    group_label = {'all': 'ALL (1–511)', 'seen': f'SEEN ({TRAIN_MIN_SRC}–{TRAIN_MAX_SRC})',
                   'unseen': f'UNSEEN (1–{TRAIN_MIN_SRC-1} + {TRAIN_MAX_SRC+1}–511)'}
    print(f'\n{"="*140}')
    print(f'  {title}  |  strategy = {strategy}  |  {group_label[group]}')
    print(f'  Errors: prediction - manual.  Negative bias → model predicts earlier than manual label.')
    print(f'{"="*140}')
    header = (f'{"Experiment":30s}  {"Max Err":>10s}  {"MAE":>10s}  '
              f'{"Bias":>10s}  {"Var (s²)":>10s}  '
              f'{"Precision":>10s}  {"Recall":>10s}  {"F1(%)":>10s}  {"n_src":>6s}')
    print(header)
    print('-' * 140)
    rows = []
    for exp_key, exp_label in experiments:
        key = f'{exp_key}_{strategy}'
        if key not in all_metrics:
            print(f'{exp_label:30s}  {"MISSING":>10s}')
            continue
        subset = filter_src(all_metrics[key], group)
        if len(subset) == 0:
            print(f'{exp_label:30s}  {"NO DATA":>10s}')
            continue
        agg = aggregate(subset)
        print(f'{exp_label:30s}  {agg["max_err_s"]:10.2e}  {agg["mae_s"]:10.2e}  '
              f'{agg["mean_err_s"]:10.2e}  {agg["var_s2"]:10.2e}  '
              f'{agg["precision"]:10.4f}  {agg["recall"]:10.4f}  {agg["f1"]*100:10.2f}  {agg["n_src"]:6d}')
        rows.append((exp_label, agg))
    return rows

In [7]:
# =========================================================================
# Table: best_s3  (UNSEEN)
# =========================================================================
for group in ['unseen']:
    _ = print_table('Ablation on Ex-Vivo Data', STRATEGY, EXPERIMENTS, ALL_METRICS, group=group)


  Ablation on Ex-Vivo Data  |  strategy = best_s3  |  UNSEEN (1–127 + 385–511)
  Errors: prediction - manual.  Negative bias → model predicts earlier than manual label.
Experiment                         Max Err         MAE        Bias    Var (s²)   Precision      Recall       F1(%)   n_src
--------------------------------------------------------------------------------------------------------------------------------------------
Full                              4.48e-06    1.43e-06   -1.12e-06    1.79e-12      0.9745      0.9942       98.42     128
No Warmup                         4.55e-06    1.37e-06   -9.93e-07    1.94e-12      0.9760      0.9922       98.40     128
No TOF                            5.84e-06    1.92e-06   -1.58e-06    2.88e-12      0.9666      0.9931       97.96     128
No DomainRand                     5.20e-06    1.48e-06   -9.23e-07    2.64e-12      0.9757      0.9907       98.31     128
No Aug                            6.22e-06    1.90e-06   -1.57e-06    3.10

---
## Notes
- **No fine-tuning**: models evaluated directly from pretraining `best_s3` checkpoints.
- **SEEN/UNSEEN split** matches the fine-tuning training range for fair comparison.
- Ex-vivo data from `manual_seg_beef`: 255 odd src × 64 slices each.
- Inference uses slice 32 for all sources.